# NB13b: Spatial Confusion-Matrix Visualization v2

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


Renders the per-building OOF predictions (produced by NB07/08/09 retrofits) as spatial maps colored by confusion-matrix class (TP / TN / FP / FN). Figures are both shown inline and saved to disk under `RESULTS_ROOT/nb11b/`.

Sections:
- **M1 DISCOVERY** — find OOF parquets (same glob as NB11a)
- **M2 LOAD + DEDUP** — concat + keep latest per `model_id`; compute per-model AUC
- **M3 MODEL SELECTION** — pick which models + cities to plot via config at top
- **M4 PER-MODEL SPATIAL MAPS** — one multi-panel figure per model (city grid). Main thesis figure.
- **M4b FULL-CITY VIEWS (best models only)** — per best model (AUC > threshold), multi-city panels showing ALL Overture buildings; blue = no UNOSAT label, CM colors for labeled.
- **M5 SINGLE-CITY MODEL STRIPS** — per city, grid of N top models side-by-side. Answers "do different models fail differently in the same place?"
- **M6 CROSS-MODEL OVERLAY** — two/three selected models overlaid. Shows correlated vs independent errors
- **M7 HARD-FOR-ALL SPATIAL** — buildings every model misclassifies. Aimaiti 2022 framing
- **M8 SUMMARY** — list of saved paths

Conventions (same as other notebooks):
- TP = red (#d62728 — Destroyed, model correct)
- TN = green (#2ca02c — Not destroyed, model correct)
- FP = pink (#ff9ecb — model hallucinated damage)
- FN = gold (#ffd700 — model missed damage)
- Draw order TN → FP → FN → TP so that errors and TPs sit on top of the TN background

NB11c (future) handles cross-validation / statistical rigor; this notebook is visualization only.

# CONFIG + GLOBAL SETUP

In [ ]:
# @title CELL 3: NB11b CONFIG + GLOBAL SETUP
import sys, os
from pathlib import Path

import platform, os
if platform.system() == "Windows":
    _setup = r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py"
elif os.path.exists("/content/drive_f"):
    _setup = "/content/drive_f/masterthesis/notebooks/global_setup.py"
else:
    _setup = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py"
with open(_setup) as f:
    exec(f.read())

# --- tier / city selection (match upstream notebooks) ------------------
TIER_SELECTION = [0, 1, 2]
CITY_SELECTION = None
REQUIRE_UNOSAT = True
RANDOM_STATE = 42
TOP_N_SPATIAL = 20

# --- OOF discovery roots ---------------------------------------------
OOF_SEARCH_ROOTS = [RESULTS_ROOT]
try:
    _nb09d_root = CONTENT_LOCAL / 'nb09d_dl_experiments'
    if _nb09d_root.exists():
        OOF_SEARCH_ROOTS.append(_nb09d_root)
except NameError:
    pass

# --- NB11 hyperparameter-tuning OOF parquets (added in v6) ----------
try:
    _nb11_root = OUTPUTS_DIR / 'NB11_V2' / 'oof'
    if _nb11_root.exists():
        OOF_SEARCH_ROOTS.append(_nb11_root)
except NameError:
    pass

# --- plot selection / config -----------------------------------------
# Which cities to plot. None = all cities present in the OOF data.
CITIES_TO_PLOT = None

# Which models to plot. Options:
#   None                 -> top N by AUC (see MAX_MODELS_TO_PLOT)
#   'all'                -> every headline (non-leaky, non-degenerate) model
#   list of model_ids    -> only those exact models
MODELS_TO_PLOT = None

# Cap on number of models in M4 (one multi-city figure per model)
MAX_MODELS_TO_PLOT = 64

# In M5 (single-city model strip), how many top models per city
TOP_N_PER_CITY = 6

# In M6 cross-model overlay, which pairs to plot. List of (model_a, model_b) tuples.
# If empty: picks two extreme models (highest and lowest AUC in headline set).
CROSS_MODEL_PAIRS = []

# Plot style:
#   'centroid' -> scatter of building centroids (fast, works with any GeoDataFrame)
#   'polygon'  -> fill building polygons (slower, polished for thesis figures)
PLOT_STYLE = 'centroid'

# Polygon rendering only: edge width, fill alpha
POLYGON_EDGE_WIDTH = 0.15
POLYGON_ALPHA = 0.75

# Exclude these variant markers from automatic top-N selection (but still plottable by name)
LEAKAGE_VARIANT_MARKERS = ['leakage_demo', 'random_cv', 'geographic_70_30']

# Hard-for-all threshold (M7); should match NB11a setting
HARD_FOR_ALL_THRESHOLD = 0.90

# M4b: full-city views — only render models with AUC above this threshold
M4B_AUC_THRESHOLD = 0.75

print(f"  OOF search roots:")
for r in OOF_SEARCH_ROOTS:
    print(f"    {r}")
print(f"  TIER_SELECTION:        {TIER_SELECTION}")
print(f"  PLOT_STYLE:            {PLOT_STYLE}")
print(f"  MAX_MODELS_TO_PLOT:    {MAX_MODELS_TO_PLOT}")
print(f"  TOP_N_PER_CITY:        {TOP_N_PER_CITY}")
print(f"  HARD_FOR_ALL_THRESHOLD: {HARD_FOR_ALL_THRESHOLD}")
print(f"  M4B_AUC_THRESHOLD:      {M4B_AUC_THRESHOLD}  (M4b renders only models with AUC > threshold)")


# CELL S0: BUILDINGS + GEOMETRIES + OUTPUT PATHS
Loads building metadata (tabular) and per-city polygon geometries (GeoDataFrame). Computes centroids if not already present. Sets up output dir + save helpers.

In [ ]:
# @title CELL S0: SETUP (A/B GROUPS + MODALITY + ACCUMULATORS)
import json, gc, time, re
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime as _dt
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("CELL S0: NB11b SPATIAL VIZ -- SETUP")
print("=" * 70)

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import oof_loader
import importlib
importlib.reload(oof_loader)
from oof_loader import (
    GROUP_A, GROUP_B, SAMPLE_ID_COL,
    discover_oof_parquets, load_all_oof,
    load_metadata_both, check_y_true_consistency,
    build_model_sanity, build_gdf_by_city,
)

CITIES_TO_PROCESS, _battle_dates = resolve_cities(
    tier_selection=TIER_SELECTION,
    city_selection=[CITY_SELECTION] if isinstance(CITY_SELECTION, str) else CITY_SELECTION,
    require_unosat=REQUIRE_UNOSAT,
)
_tiers = TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2]
DATASET_ROOT_V3 = STACK_DIR / 'dataset' / 'v3'

meta = load_metadata_both(
    dataset_root_v2=DATASET_ROOT_V2,
    dataset_root_v3=DATASET_ROOT_V3,
    tiers=_tiers,
    cities=CITIES_TO_PROCESS,
    load_tier_parquets_fn=load_tier_parquets,
)
df_bldg = meta['buildings_df']
df_pts  = meta['points_df']
coord_maps = {GROUP_A: meta['coord_map_A'], GROUP_B: meta['coord_map_B']}

# --- modality classifier ------------------------------------------------
def classify_modality(model_id, variant_id):
    combined = (str(model_id) + ' ' + str(variant_id)).lower()
    # --- NB11 explicit parquet -> modality lookup (added in v6) --------
    # NB11 model_ids look like 'NB11<a-f>_<tag>__<manifest_key>__<clf>-Optuna'.
    # We check manifest_key tokens FIRST so they take precedence over the
    # token-substring matchers below (which would otherwise miss 'fusion_*').
    _nb11_modality = {
        # SAR-only
        'card_drop':                  'SAR_card_only',
        'prepost_single_card':        'SAR_card_only',
        'coh_drop':                   'SAR_coh_only',
        'rolling_accum_coh':          'SAR_coh_only',
        # SAR + COH fused (no MS)
        'fusion_card_cohdrop':        'SAR_card+coh',
        # multimodal (SAR + MS, +/- COH)
        'fusion_ms_card_cohdrop':     'multimodal_all',
        'fusion_indices_card_cohdrop':'multimodal_all',
        'fusion_ms_cohdrop':          'multimodal_SAR+MS',
        'fusion_composite_cohdrop':   'multimodal_all',
        'fusion_composite_blockstats':'multimodal_all',
        'block_stats':                'multimodal_all',
        'rolling_stats_roll3':        'multimodal_all',
        'rolling_stats_roll7':        'multimodal_all',
        'rolling_stats_roll13':       'multimodal_all',
        # MS-only
        'ms_change':                  'MS_only',
        'composite_prepost_bands':    'MS_only',
        'composite_vs_scenes_bands':  'MS_only',
        # land-use (own family)
        'lu_change':                  'other',
        'composite_prepost_landuse':  'other',
        'composite_vs_scenes_landuse':'other',
    }
    if 'nb11' in combined:
        for _key, _mod in _nb11_modality.items():
            if _key in combined:
                return _mod
    if any(x in combined for x in ['coh_drop', 'coh_running', 'coh_max_drop']):
        return 'SAR_coh_unsupervised'
    if 'logratio' in combined:
        if any(x in combined for x in ['ndvi', 'savi', 'bsi', 'nbr', 'ndbi']):
            return 'MS_unsupervised'
        return 'SAR_unsupervised'
    if 'card_only' in combined or 'sensor=card;' in combined or combined.endswith('sensor=card'):
        return 'SAR_card_only'
    if 'coh_only' in combined or 'sensor=coh;' in combined or combined.endswith('sensor=coh'):
        return 'SAR_coh_only'
    if 'card+coh' in combined or 'sensor=card+coh' in combined:
        return 'SAR_card+coh'
    if 'ms_only' in combined or 'sensor=ms' in combined:
        return 'MS_only'
    if 'nb08b' in combined:
        if 'rgb' in combined and ('card' in combined or 'coh' in combined):
            return 'multimodal_SAR+MS'
        if 'rgb' in combined:
            return 'MS_only'
    if 'all_multimodal' in combined or 'all features' in combined:
        return 'multimodal_all'
    if 'parquet=f7' in combined or 'parquet=a17' in combined:
        return 'multimodal_all'
    if 'sar_only' in combined:
        return 'SAR_all'
    if 'sar + ms + coh' in combined:
        return 'multimodal_all'
    if 'sar + ms' in combined:
        return 'multimodal_SAR+MS'
    return 'other'

MODALITY_FAMILIES = {
    'SAR': ['SAR_card_only', 'SAR_coh_only', 'SAR_card+coh', 'SAR_all',
            'SAR_coh_unsupervised', 'SAR_unsupervised'],
    'MS':  ['MS_only', 'MS_unsupervised'],
    'multimodal': ['multimodal_all', 'multimodal_SAR+MS'],
}
def modality_family(mod):
    for fam, members in MODALITY_FAMILIES.items():
        if mod in members:
            return fam
    return 'other'

# --- accumulator detector -----------------------------------------------
ACCUM_PATTERNS = [
    'coh_drop', 'coh_running_min', 'coh_max_drop', 'drop_count',
    'swir_rise', 'nbr_anomaly', 'mahalanobis', 'dnbr', 'rbr',
    'lu_change', 'loss_count', 'loss_fraction',
    'accum', 'a14', 'a19', 'a20', 'a21', 'a22',
]
def is_accumulator_model(model_id, variant_id):
    combined = (str(model_id) + ' ' + str(variant_id)).lower()
    return any(p in combined for p in ACCUM_PATTERNS)

# --- discovery + loading ------------------------------------------------
inv_df = discover_oof_parquets(OOF_SEARCH_ROOTS)
oof_A, oof_B = load_all_oof(inv_df)

groups_loaded = {}
if oof_A is not None and len(oof_A) > 0:
    groups_loaded[GROUP_A] = oof_A
if oof_B is not None and len(oof_B) > 0:
    groups_loaded[GROUP_B] = oof_B

# --- model sanity with tags per group ------------------------------------
model_sanity_all = {}
headline_all = {}

for grp_name, grp_oof in groups_loaded.items():
    check_y_true_consistency(grp_oof, grp_name)
    ms = build_model_sanity(grp_oof, grp_name, LEAKAGE_VARIANT_MARKERS)
    ms['modality'] = ms.apply(lambda r: classify_modality(r['model_id'], r['variant_id']), axis=1)
    ms['modality_family'] = ms['modality'].apply(modality_family)
    ms['is_accumulator'] = ms.apply(lambda r: is_accumulator_model(r['model_id'], r['variant_id']), axis=1)
    model_sanity_all[grp_name] = ms
    headline = ms.loc[(~ms['is_leaky']) & (~ms['is_degenerate']), 'model_id'].tolist()
    headline_all[grp_name] = headline

# --- coord helpers -------------------------------------------------------
CM_COLORS = {'TP': '#d62728', 'TN': '#2ca02c', 'FP': '#ff9ecb', 'FN': '#ffd700',
             'no_unosat': '#4a90d9'}

def add_coords(df, grp_name):
    cm = coord_maps[grp_name]
    df = df.copy()
    df['coord_x'] = df[SAMPLE_ID_COL].map(lambda s: cm.get(s, (np.nan, np.nan))[0])
    df['coord_y'] = df[SAMPLE_ID_COL].map(lambda s: cm.get(s, (np.nan, np.nan))[1])
    return df.dropna(subset=['coord_x', 'coord_y'])

def get_bg_df(grp_name):
    if grp_name == GROUP_A:
        return df_bldg, 'building_id', 'centroid_x', 'centroid_y'
    else:
        return df_pts, 'point_id', 'x_utm', 'y_utm'

# --- output ---------------------------------------------------------------
OUT_DIR = RESULTS_ROOT / 'nb11b'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_result(data, name, cell_id, fmt='csv'):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    if fmt == 'csv' and isinstance(data, pd.DataFrame):
        path = cell_dir / f"{name}_{ts}.csv"
        data.to_csv(path, index=False)
    elif fmt == 'json':
        path = cell_dir / f"{name}_{ts}.json"
        with open(path, 'w') as fh:
            json.dump(data, fh, indent=2, default=str)
    else:
        raise ValueError(f"Unknown fmt={fmt}")
    print(f"  Saved: {path.relative_to(OUT_DIR)} ({path.stat().st_size / 1024:.1f} KB)")
    return path

def save_fig(fig, name, cell_id, dpi=150, show=True):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    path = cell_dir / f"{name}_{ts}.png"
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    print(f"  Plot: {path.relative_to(OUT_DIR)}")
    return path

print(f"  Groups: {list(groups_loaded.keys())}")
for g, ms in model_sanity_all.items():
    hl = ms[(~ms['is_leaky']) & (~ms['is_degenerate'])]
    n_acc = int(hl['is_accumulator'].sum())
    print(f"  [{g}] {len(hl)} headline, {n_acc} accumulators")
    for fam in ['SAR', 'MS', 'multimodal', 'other']:
        sub = hl[hl['modality_family'] == fam]
        if len(sub) > 0:
            print(f"    {fam:15s}: n={len(sub):3d}  median_AUC={sub['auc'].median():.3f}  "
                  f"max={sub['auc'].max():.3f}  accum={int(sub['is_accumulator'].sum())}")
print(f"  Output: {OUT_DIR}")


# CELL S0b: SPATIAL PLOT HELPERS
Two modes:
- `plot_cm_city(oof_city_df, gdf_city, ax, ...)` — render one city panel (centroid scatter or polygon fills)
- `plot_cm_grid(oof_df, gdfs, cities, ...)` — grid of city panels for one model

Colors match other notebooks' S0b cells: TP red, TN green, FP pink, FN gold.

In [ ]:
# @title CELL S0b: SPATIAL PLOT HELPER
def plot_city_panel(ax, mdf, city, grp_name, show_bg=False, title_extra=''):
    """Plot one city CM scatter on ax. mdf must have coord_x, coord_y, cm_class."""
    cdf = mdf[mdf['city'] == city]
    if show_bg:
        bg_df, bg_id, bg_x, bg_y = get_bg_df(grp_name)
        if bg_df is not None:
            all_city = bg_df[bg_df['city'] == city]
            no_label = all_city[~all_city[bg_id].isin(cdf[SAMPLE_ID_COL])]
            if len(no_label) > 0:
                ax.scatter(no_label[bg_x], no_label[bg_y],
                           c=CM_COLORS['no_unosat'], s=1, alpha=0.15, edgecolors='none')
    for cm_cls in ['TN', 'FP', 'FN', 'TP']:
        sub = cdf[cdf['cm_class'] == cm_cls]
        if len(sub) == 0:
            continue
        sz = 2 if cm_cls == 'TN' else 6
        alpha = 0.3 if cm_cls == 'TN' else 0.8
        ax.scatter(sub['coord_x'], sub['coord_y'],
                   c=CM_COLORS[cm_cls], s=sz, alpha=alpha, edgecolors='none')
    cm_c = cdf['cm_class'].value_counts()
    ax.set_title(f"{city}  (n={len(cdf):,}){title_extra}\n"
                 f"TP={int(cm_c.get('TP',0))} FN={int(cm_c.get('FN',0))} "
                 f"FP={int(cm_c.get('FP',0))} TN={int(cm_c.get('TN',0))}", fontsize=7)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=5)

def make_city_grid(mdf, cities, grp_name, suptitle, cell_id, fname, show_bg=False):
    """Create multi-panel city grid figure. Returns fig path or None."""
    cities_ok = [c for c in cities if c in mdf['city'].values]
    if not cities_ok:
        return None
    ncols = min(3, len(cities_ok))
    nrows = (len(cities_ok) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 4.5 * nrows), squeeze=False)
    fig.suptitle(suptitle, fontsize=10)
    for ci, city in enumerate(cities_ok):
        plot_city_panel(axes[ci // ncols][ci % ncols], mdf, city, grp_name, show_bg=show_bg)
    for j in range(len(cities_ok), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')
    handles = [Patch(facecolor=CM_COLORS[c], label=c) for c in ['TP', 'FN', 'FP', 'TN']]
    if show_bg:
        handles.insert(0, Patch(facecolor=CM_COLORS['no_unosat'], label='no UNOSAT'))
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), fontsize=8)
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    p = save_fig(fig, fname, cell_id)
    plt.close(fig)
    return p

print("  Plot helpers ready: plot_city_panel(), make_city_grid()")


# CELL M1: DISCOVERY — GLOB OOF PARQUETS
Same pattern as NB11a M1. Self-contained so NB11b can run without NB11a.

In [ ]:
# @title CELL M1: DISCOVERY (done in S0)
print("=" * 70)
print("CELL M1: DISCOVERY (handled in S0)")
print("=" * 70)
print(f"  {len(inv_df)} parquets discovered")
save_result(inv_df.drop(columns=['path'], errors='ignore'), 'M1_oof_inventory', 'cell_m1')


# CELL M2: LOAD + DEDUP + PER-MODEL AUC
Loads latest per `model_id`, computes per-model AUC, flags leaky + degenerate.

In [ ]:
# @title CELL M2: MODEL SANITY (done in S0)
print("=" * 70)
print("CELL M2: MODEL SANITY (handled in S0)")
print("=" * 70)
for grp_name, ms in model_sanity_all.items():
    save_result(ms, f'M2_model_sanity_{grp_name}', 'cell_m2')
    print(f"  [{grp_name}] {len(ms)} models saved")


# CELL M3: MODEL SELECTION
Resolves `MODELS_TO_PLOT` from config. If `None`, picks top `MAX_MODELS_TO_PLOT` by AUC from the headline set (non-leaky, non-degenerate).  If `'all'`, uses every headline model. If a list, uses exactly those.

In [ ]:
# @title CELL M3: SELECT MODELS (PER GROUP + MODALITY)
print("=" * 70)
print("CELL M3: SELECT HEADLINE MODELS")
print("=" * 70)

TOP_N = globals().get('TOP_N_SPATIAL', 20)
selected_all = {}

for grp_name in groups_loaded:
    ms = model_sanity_all[grp_name]
    headline = ms[(~ms['is_leaky']) & (~ms['is_degenerate'])].copy()
    headline_sorted = headline.sort_values('auc', ascending=False)
    N = min(TOP_N, len(headline_sorted))
    selected = headline_sorted.head(N)['model_id'].tolist()
    selected_all[grp_name] = selected
    print(f"\n  [{grp_name}] {N} models selected")

    # breakdown
    for fam in ['SAR', 'MS', 'multimodal', 'other']:
        sub = headline_sorted[headline_sorted['modality_family'] == fam]
        if len(sub) > 0:
            best = sub.iloc[0]
            print(f"    {fam:15s}: {len(sub):3d} total  best={best['auc']:.3f} ({best['model_id']})")

    # accumulators
    acc = headline_sorted[headline_sorted['is_accumulator']]
    if len(acc) > 0:
        best_acc = acc.iloc[0]
        print(f"    {'accumulators':15s}: {len(acc):3d} total  best={best_acc['auc']:.3f} ({best_acc['model_id']})")

    save_result(selected_all[grp_name], f'M3_selected_{grp_name}', 'cell_m3', fmt='json')


# CELL M4: PER-MODEL SPATIAL CM MAPS
One multi-panel figure per selected model. Each panel is a city showing TP/TN/FP/FN colored buildings.  Both displayed inline and saved to `cell_m4/`.

In [ ]:
# @title CELL M4: SPATIAL CONFUSION MAPS (PER GROUP)
print("=" * 70)
print("CELL M4: SPATIAL CONFUSION MAPS")
print("=" * 70)

fig_index = []
for grp_name, grp_oof in groups_loaded.items():
    print(f"\n  === {grp_name} ===")
    sel_mids = selected_all.get(grp_name, [])
    if not sel_mids:
        continue
    cities = sorted(grp_oof['city'].unique())
    for mid in sel_mids:
        mdf = grp_oof[grp_oof['model_id'] == mid].copy()
        mdf = add_coords(mdf, grp_name)
        if len(mdf) == 0:
            continue
        ms_row = model_sanity_all[grp_name]
        auc = ms_row.loc[ms_row['model_id'] == mid, 'auc'].iloc[0]
        mod = ms_row.loc[ms_row['model_id'] == mid, 'modality_family'].iloc[0]
        is_acc = bool(ms_row.loc[ms_row['model_id'] == mid, 'is_accumulator'].iloc[0])
        cm = mdf['cm_class'].value_counts()
        tag = f" [ACCUM]" if is_acc else ""
        title = (f"{mid} [{grp_name}] [{mod}]{tag}  AUC={auc:.3f}\n"
                 f"TP={int(cm.get('TP',0))} FN={int(cm.get('FN',0))} "
                 f"FP={int(cm.get('FP',0))} TN={int(cm.get('TN',0))}")
        fname = mid.replace('/', '_').replace(' ', '_').replace(':', '_')
        p = make_city_grid(mdf, cities, grp_name, title, 'cell_m4', f'M4_{fname}_{grp_name}')
        if p:
            fig_index.append({'model_id': mid, 'group': grp_name, 'modality': mod,
                              'is_accumulator': is_acc, 'auc': auc})

if fig_index:
    save_result(pd.DataFrame(fig_index), 'M4_figures_index', 'cell_m4')
print(f"\n  M4 DONE: {len(fig_index)} figures")


# CELL M4b: FULL-CITY VIEWS FOR BEST MODELS (AUC > M4B_AUC_THRESHOLD)
Same layout as M4 (multi-city panels per model) but with every Overture building shown.
- BLUE dots/polygons = building is in the Overture city geojson but has no row in the OOF parquet (i.e. no UNOSAT label, or dropped during feature prep for this model)
- TP/TN/FP/FN = as elsewhere

This shows the **labeled subset in geographic context** — makes UNOSAT's coverage gaps visible, which matters for the thesis "UNOSAT = Achilles Heel" framing.

In [ ]:
# @title CELL M4b: FULL-CITY VIEWS (AUC > threshold, PER GROUP)
print("=" * 70)
print(f"CELL M4b: FULL-CITY VIEWS  (AUC > {M4B_AUC_THRESHOLD})")
print("=" * 70)

fig_index_4b = []
for grp_name, grp_oof in groups_loaded.items():
    print(f"\n  === {grp_name} ===")
    ms = model_sanity_all[grp_name]
    best = ms[(~ms['is_leaky']) & (~ms['is_degenerate']) & (ms['auc'] > M4B_AUC_THRESHOLD)]
    best = best.sort_values('auc', ascending=False)
    print(f"  {len(best)} models above threshold")
    if len(best) == 0:
        continue
    cities = sorted(grp_oof['city'].unique())
    for _, row in best.iterrows():
        mid = row['model_id']
        mdf = grp_oof[grp_oof['model_id'] == mid].copy()
        mdf = add_coords(mdf, grp_name)
        if len(mdf) == 0:
            continue
        tag = " [ACCUM]" if row['is_accumulator'] else ""
        title = (f"{mid} [{grp_name}]{tag}  AUC={row['auc']:.3f}\n"
                 f"TP={int(row['TP'])} FN={int(row['FN'])} FP={int(row['FP'])} TN={int(row['TN'])}  "
                 f"+ background")
        fname = mid.replace('/', '_').replace(' ', '_').replace(':', '_')
        p = make_city_grid(mdf, cities, grp_name, title, 'cell_m4b',
                           f'M4b_{fname}_{grp_name}', show_bg=True)
        if p:
            fig_index_4b.append({'model_id': mid, 'group': grp_name,
                                 'modality': row['modality_family'],
                                 'is_accumulator': row['is_accumulator'], 'auc': row['auc']})

if fig_index_4b:
    save_result(pd.DataFrame(fig_index_4b), 'M4b_figures_index', 'cell_m4b')
print(f"\n  M4b DONE: {len(fig_index_4b)} figures")


# CELL M5: SINGLE-CITY MODEL COMPARISON STRIPS
One figure per city; each figure contains `TOP_N_PER_CITY` model panels side-by-side. Useful for thesis "model A vs model B on Mariupol" comparisons.

In [ ]:
# @title CELL M5: SINGLE-CITY STRIPS (PER GROUP, TOP PER MODALITY)
print("=" * 70)
print(f"CELL M5: SINGLE-CITY STRIPS")
print("=" * 70)

saved_m5 = []
for grp_name, grp_oof in groups_loaded.items():
    print(f"\n  === {grp_name} ===")
    ms = model_sanity_all[grp_name]
    # pick best per modality family + best accumulator
    picks = []
    for fam in ['SAR', 'MS', 'multimodal']:
        sub = ms[(ms['modality_family'] == fam) & (~ms['is_leaky']) & (~ms['is_degenerate'])]
        if len(sub) > 0:
            picks.append(sub.nlargest(1, 'auc').iloc[0]['model_id'])
    acc = ms[(ms['is_accumulator']) & (~ms['is_leaky']) & (~ms['is_degenerate'])]
    if len(acc) > 0:
        best_acc = acc.nlargest(1, 'auc').iloc[0]['model_id']
        if best_acc not in picks:
            picks.append(best_acc)
    # fill remaining slots from top AUC
    headline = ms[(~ms['is_leaky']) & (~ms['is_degenerate'])].sort_values('auc', ascending=False)
    for _, r in headline.iterrows():
        if len(picks) >= TOP_N_PER_CITY:
            break
        if r['model_id'] not in picks:
            picks.append(r['model_id'])
    print(f"  Strip models ({len(picks)}): {picks}")

    cities = sorted(grp_oof['city'].unique())
    for city in cities:
        ncols = min(3, len(picks))
        nrows = (len(picks) + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 4.5 * nrows), squeeze=False)
        fig.suptitle(f'{city} [{grp_name}] -- top models by modality', fontsize=11)
        for i, mid in enumerate(picks):
            ax = axes[i // ncols][i % ncols]
            mdf = grp_oof[(grp_oof['model_id'] == mid) & (grp_oof['city'] == city)].copy()
            mdf = add_coords(mdf, grp_name)
            if len(mdf) == 0:
                ax.text(0.5, 0.5, f'{mid}\nno data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=7, color='grey')
                ax.set_xticks([]); ax.set_yticks([])
                continue
            r = ms[ms['model_id'] == mid].iloc[0]
            tag = " ACC" if r['is_accumulator'] else ""
            plot_city_panel(ax, mdf, city, grp_name,
                            title_extra=f"\n{r['modality_family']}{tag} AUC={r['auc']:.3f}")
        handles = [Patch(facecolor=CM_COLORS[c], label=c) for c in ['TP', 'FN', 'FP', 'TN']]
        fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=8)
        for j in range(len(picks), nrows * ncols):
            axes[j // ncols][j % ncols].axis('off')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        save_fig(fig, f'M5_strip_{city}_{grp_name}', 'cell_m5', show=True)
        plt.close(fig)
        saved_m5.append({'city': city, 'group': grp_name})

print(f"\n  M5 DONE: {len(saved_m5)} strip figures")


# CELL M6: CROSS-MODEL OVERLAY — WHERE DO MODELS DIFFER?
For a pair of models (A, B), splits buildings into five sets:
- `both_correct` — both TN or both TP (grey, background)
- `only_A_FP` or `only_A_FN` — A made an error B didn't (cyan)
- `only_B_FP` or `only_B_FN` — B made an error A didn't (magenta)
- `both_wrong` — both erred (black)

Answers the "are errors correlated or independent?" question spatially. Complements NB11a's numerical Jaccard.

In [ ]:
# @title CELL M6: CROSS-MODEL OVERLAY (PER GROUP)
print("=" * 70)
print("CELL M6: CROSS-MODEL OVERLAY")
print("=" * 70)

OVERLAY_COLORS = {
    'both_correct': '#d0d0d0', 'only_A': '#00bcd4',
    'only_B': '#e91e63', 'both_wrong': '#000000',
}

saved_m6 = []
for grp_name, grp_oof in groups_loaded.items():
    print(f"\n  === {grp_name} ===")
    ms = model_sanity_all[grp_name]
    headline = ms[(~ms['is_leaky']) & (~ms['is_degenerate'])].sort_values('auc', ascending=False)
    if len(headline) < 2:
        continue

    # auto-pairs: best SAR vs best MS, best overall vs worst overall, best accum vs best non-accum
    pairs = list(CROSS_MODEL_PAIRS)
    best_sar = headline[headline['modality_family'] == 'SAR']
    best_ms = headline[headline['modality_family'] == 'MS']
    if len(best_sar) > 0 and len(best_ms) > 0:
        pairs.append((best_sar.iloc[0]['model_id'], best_ms.iloc[0]['model_id']))
    if len(headline) >= 2:
        pairs.append((headline.iloc[0]['model_id'], headline.iloc[-1]['model_id']))
    best_acc = headline[headline['is_accumulator']]
    best_nonacc = headline[~headline['is_accumulator']]
    if len(best_acc) > 0 and len(best_nonacc) > 0:
        pairs.append((best_acc.iloc[0]['model_id'], best_nonacc.iloc[0]['model_id']))
    # deduplicate
    seen = set()
    unique_pairs = []
    for a, b in pairs:
        key = tuple(sorted([a, b]))
        if key not in seen and a != b:
            seen.add(key)
            unique_pairs.append((a, b))

    cities = sorted(grp_oof['city'].unique())
    for (model_a, model_b) in unique_pairs:
        oof_a = grp_oof[grp_oof['model_id'] == model_a][[SAMPLE_ID_COL, 'city', 'y_true', 'cm_class']]
        oof_b = grp_oof[grp_oof['model_id'] == model_b][[SAMPLE_ID_COL, 'city', 'y_true', 'cm_class']]
        merged = oof_a.merge(oof_b, on=[SAMPLE_ID_COL, 'city', 'y_true'], suffixes=('_A', '_B'))
        if len(merged) == 0:
            continue
        a_ok = merged['cm_class_A'].isin(['TP', 'TN'])
        b_ok = merged['cm_class_B'].isin(['TP', 'TN'])
        merged['overlay'] = 'both_correct'
        merged.loc[(~a_ok) & b_ok, 'overlay'] = 'only_A'
        merged.loc[a_ok & (~b_ok), 'overlay'] = 'only_B'
        merged.loc[(~a_ok) & (~b_ok), 'overlay'] = 'both_wrong'

        counts = merged['overlay'].value_counts()
        mod_a = ms.loc[ms['model_id'] == model_a, 'modality_family'].iloc[0]
        mod_b = ms.loc[ms['model_id'] == model_b, 'modality_family'].iloc[0]
        print(f"\n  A={model_a} [{mod_a}] vs B={model_b} [{mod_b}]  (n={len(merged):,})")
        for cat in ['both_correct', 'only_A', 'only_B', 'both_wrong']:
            n = int(counts.get(cat, 0))
            print(f"    {cat:18s} {n:7d}  ({100*n/len(merged):5.1f}%)")

        merged = add_coords(merged, grp_name)
        cities_ok = [c for c in cities if c in merged['city'].values]
        if not cities_ok:
            continue
        ncols = min(3, len(cities_ok))
        nrows = (len(cities_ok) + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 4.5 * nrows), squeeze=False)
        fig.suptitle(f'M6 overlay [{grp_name}]:  A={model_a} [{mod_a}]  vs  B={model_b} [{mod_b}]',
                     fontsize=9)
        for ci, city in enumerate(cities_ok):
            ax = axes[ci // ncols][ci % ncols]
            cdf = merged[merged['city'] == city]
            for cat in ['both_correct', 'only_A', 'only_B', 'both_wrong']:
                sub = cdf[cdf['overlay'] == cat]
                if len(sub) == 0:
                    continue
                sz = 2 if cat == 'both_correct' else 8
                alpha = 0.2 if cat == 'both_correct' else 0.85
                ax.scatter(sub['coord_x'], sub['coord_y'],
                           c=OVERLAY_COLORS[cat], s=sz, alpha=alpha, edgecolors='none')
            ax.set_aspect('equal'); ax.tick_params(labelsize=5)
            cc = cdf['overlay'].value_counts()
            ax.set_title(f"{city}  wrong_A={int(cc.get('only_A',0))} wrong_B={int(cc.get('only_B',0))} "
                         f"both_wrong={int(cc.get('both_wrong',0))}", fontsize=6)
        for j in range(len(cities_ok), nrows * ncols):
            axes[j // ncols][j % ncols].axis('off')
        handles = [Patch(facecolor=OVERLAY_COLORS[c], label=c) for c in ['both_correct', 'only_A', 'only_B', 'both_wrong']]
        fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=7)
        plt.tight_layout(rect=[0, 0.03, 1, 0.93])
        fa = model_a.replace('/','_').replace(' ','_')
        fb = model_b.replace('/','_').replace(' ','_')
        save_fig(fig, f'M6_overlay_{fa}_vs_{fb}_{grp_name}', 'cell_m6')
        plt.close(fig)
        saved_m6.append({'model_a': model_a, 'model_b': model_b, 'group': grp_name})

print(f"\n  M6 DONE: {len(saved_m6)} overlay figures")


# CELL M7: HARD-FOR-ALL SPATIAL MAP
Computes per-building error rate across headline models, colors by:
- `hard_missed`  — y_true=1, >= threshold models got it wrong  → gold (suspected signal-limited)
- `hard_phantom` — y_true=0, >= threshold models got it wrong  → red (suspected UNOSAT false negative, Aimaiti 2022 framing)
- `easy_correct` — < threshold models got it wrong  → grey background

Spatial clustering of these two categories is thesis-relevant: do they cluster in specific landuse zones / city blocks?

In [ ]:
# @title CELL M7: HARD-FOR-ALL SPATIAL MAP (PER GROUP)
print("=" * 70)
print(f"CELL M7: HARD-FOR-ALL SPATIAL  (threshold={HARD_FOR_ALL_THRESHOLD})")
print("=" * 70)

CAT_COLORS = {'easy_correct': '#cccccc',
              'hard_missed':  '#ffd700',
              'hard_phantom': '#d62728'}

for grp_name, grp_oof in groups_loaded.items():
    print(f"\n  === {grp_name} ===")
    ms = model_sanity_all[grp_name]
    headline_mids = set(ms[(~ms['is_leaky']) & (~ms['is_degenerate'])]['model_id'])
    headline_oof = grp_oof[grp_oof['model_id'].isin(headline_mids)].copy()
    print(f"  Headline models: {len(headline_mids)}")

    per_bldg = headline_oof.groupby([SAMPLE_ID_COL, 'city', 'y_true']).agg(
        n_models=('cm_class', 'count'),
        n_errors=('cm_class', lambda s: s.isin(['FP', 'FN']).sum()),
    ).reset_index()
    per_bldg['error_rate'] = per_bldg['n_errors'] / per_bldg['n_models'].replace(0, np.nan)

    min_raters = max(3, int(0.1 * len(headline_mids)))
    per_bldg_qual = per_bldg[per_bldg['n_models'] >= min_raters].copy()
    print(f"  Samples with >= {min_raters} model votes: {len(per_bldg_qual):,}")

    hard = per_bldg_qual['error_rate'] >= HARD_FOR_ALL_THRESHOLD
    per_bldg_qual['category'] = 'easy_correct'
    per_bldg_qual.loc[hard & (per_bldg_qual['y_true'] == 1), 'category'] = 'hard_missed'
    per_bldg_qual.loc[hard & (per_bldg_qual['y_true'] == 0), 'category'] = 'hard_phantom'

    cat_counts = per_bldg_qual['category'].value_counts()
    print(f"\n  Categories:")
    for c in ['easy_correct', 'hard_missed', 'hard_phantom']:
        n = int(cat_counts.get(c, 0))
        pct = 100 * n / len(per_bldg_qual) if len(per_bldg_qual) else 0
        print(f"    {c:15s}: {n:7d}  ({pct:5.2f}%)")

    save_result(per_bldg_qual, f'M7_per_sample_error_rate_{grp_name}', 'cell_m7')

    # spatial plot
    coord_map = coord_maps[grp_name]
    per_bldg_qual['coord_x'] = per_bldg_qual[SAMPLE_ID_COL].map(lambda s: coord_map.get(s, (np.nan, np.nan))[0])
    per_bldg_qual['coord_y'] = per_bldg_qual[SAMPLE_ID_COL].map(lambda s: coord_map.get(s, (np.nan, np.nan))[1])
    per_bldg_qual = per_bldg_qual.dropna(subset=['coord_x', 'coord_y'])

    cities_render = sorted(per_bldg_qual['city'].unique())
    if not cities_render:
        print(f"\n  SKIP spatial plot [{grp_name}]: no cities with enough data")
        continue

    ncols = min(3, len(cities_render))
    nrows = (len(cities_render) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 4.5 * nrows), squeeze=False)
    fig.suptitle(f'NB11b M7: Hard-for-all [{grp_name}] (error_rate >= {HARD_FOR_ALL_THRESHOLD}, '
                 f'{len(headline_mids)} models)', fontsize=11)

    draw_order = ['easy_correct', 'hard_missed', 'hard_phantom']
    for ci, city in enumerate(cities_render):
        ax = axes[ci // ncols][ci % ncols]
        plot_df = per_bldg_qual[per_bldg_qual['city'] == city]
        if len(plot_df) == 0:
            ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)
            continue
        for cat in draw_order:
            sub = plot_df[plot_df['category'] == cat]
            if len(sub) == 0:
                continue
            size = 2 if cat == 'easy_correct' else 8
            alpha = 0.35 if cat == 'easy_correct' else 0.9
            ax.scatter(sub['coord_x'], sub['coord_y'],
                       c=CAT_COLORS[cat], s=size, alpha=alpha, edgecolors='none')
        ax.set_aspect('equal')
        ax.tick_params(labelsize=6)
        cc = plot_df['category'].value_counts()
        ax.set_title(f"{city}  (n={len(plot_df):,})\n"
                     f"missed={int(cc.get('hard_missed',0))}  phantom={int(cc.get('hard_phantom',0))}",
                     fontsize=8)

    handles = [Patch(facecolor=CAT_COLORS[cat],
                     label={'easy_correct': 'Easy', 'hard_missed': 'Hard missed',
                            'hard_phantom': 'Hard phantom'}[cat])
               for cat in draw_order]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=8)
    for j in range(len(cities_render), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')
    plt.tight_layout(rect=[0, 0.02, 1, 0.97])
    save_fig(fig, f'M7_hard_for_all_spatial_{grp_name}', 'cell_m7')


# CELL M8: SUMMARY
Lists all saved figures + pointers to NB11c (cross-validation analysis, next).

In [ ]:
# @title CELL M8: ACCUMULATOR DEEP DIVE
print("=" * 70)
print("CELL M8: ACCUMULATOR DEEP DIVE")
print("=" * 70)

for grp_name, grp_oof in groups_loaded.items():
    print(f"\n  === {grp_name} ===")
    ms = model_sanity_all[grp_name]
    headline = ms[(~ms['is_leaky']) & (~ms['is_degenerate'])]
    acc = headline[headline['is_accumulator']].sort_values('auc', ascending=False)
    nonacc = headline[~headline['is_accumulator']].sort_values('auc', ascending=False)

    if len(acc) == 0:
        print(f"  [{grp_name}] no accumulator models")
        continue

    print(f"\n  Accumulators ({len(acc)}):")
    for _, r in acc.head(10).iterrows():
        print(f"    AUC={r['auc']:.3f}  [{r['modality']:25s}]  {r['model_id']}")

    print(f"\n  Non-accumulators ({len(nonacc)}):")
    for _, r in nonacc.head(5).iterrows():
        print(f"    AUC={r['auc']:.3f}  [{r['modality']:25s}]  {r['model_id']}")

    # per-city recall comparison: best accumulator vs best non-accumulator
    if len(acc) > 0 and len(nonacc) > 0:
        best_acc_mid = acc.iloc[0]['model_id']
        best_nonacc_mid = nonacc.iloc[0]['model_id']
        print(f"\n  Per-city comparison: {best_acc_mid} (accum) vs {best_nonacc_mid} (non-accum)")

        rows = []
        for city in sorted(grp_oof['city'].unique()):
            for mid, label in [(best_acc_mid, 'accumulator'), (best_nonacc_mid, 'non_accumulator')]:
                cdf = grp_oof[(grp_oof['model_id'] == mid) & (grp_oof['city'] == city)]
                cm = cdf['cm_class'].value_counts()
                tp = int(cm.get('TP', 0)); fn = int(cm.get('FN', 0))
                fp = int(cm.get('FP', 0)); tn = int(cm.get('TN', 0))
                recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
                precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
                rows.append({
                    'city': city, 'model_type': label, 'model_id': mid,
                    'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
                    'recall': round(recall, 3) if not np.isnan(recall) else np.nan,
                    'precision': round(precision, 3) if not np.isnan(precision) else np.nan,
                })

        comp = pd.DataFrame(rows)
        pivot = comp.pivot_table(index='city', columns='model_type',
                                 values=['recall', 'precision'], aggfunc='first')
        print(pivot.to_string())
        save_result(comp, f'M8_accumulator_comparison_{grp_name}', 'cell_m8')

        # spatial side-by-side: best accumulator vs best non-accumulator for 3 interesting cities
        interesting = comp[comp['model_type'] == 'accumulator'].dropna(subset=['recall'])
        interesting = interesting.sort_values('recall', ascending=False)
        show_cities = interesting['city'].head(6).tolist()

        for mid, label in [(best_acc_mid, 'accum'), (best_nonacc_mid, 'non_accum')]:
            mdf = grp_oof[grp_oof['model_id'] == mid].copy()
            mdf = add_coords(mdf, grp_name)
            fname = mid.replace('/', '_').replace(' ', '_')
            make_city_grid(mdf, show_cities, grp_name,
                           f'{mid} [{label}] [{grp_name}]', 'cell_m8',
                           f'M8_{fname}_{label}_{grp_name}')


# CELL M9: V2 vs V3 OVERALL COMPARISON
Side-by-side comparison of buildings_v2 (zonal stats) vs points_v3plus (pixel extraction).
Key question: does extraction geometry change spatial error patterns?

In [ ]:
# @title CELL M9: V2 vs V3 OVERALL COMPARISON
print("=" * 70)
print("CELL M9: V2 vs V3 OVERALL COMPARISON")
print("=" * 70)

if len(groups_loaded) < 2:
    print("  SKIP: need both groups")
else:
    # find models present in both groups
    ms_a = model_sanity_all[GROUP_A]
    ms_b = model_sanity_all[GROUP_B]
    both = ms_a[['model_id', 'auc', 'modality_family', 'is_accumulator']].merge(
        ms_b[['model_id', 'auc']], on='model_id', suffixes=('_v2', '_v3'))
    both['delta'] = both['auc_v2'] - both['auc_v3']
    both = both[(~both['auc_v2'].isna()) & (~both['auc_v3'].isna())]

    print(f"  Models in both groups: {len(both)}")
    if len(both) == 0:
        print("  No overlapping models found")
    else:
        # summary table
        rows = []
        for fam in ['SAR', 'MS', 'multimodal', 'other']:
            sub = both[both['modality_family'] == fam]
            if len(sub) == 0:
                continue
            rows.append({
                'modality': fam, 'n': len(sub),
                'v2_median': round(sub['auc_v2'].median(), 3),
                'v3_median': round(sub['auc_v3'].median(), 3),
                'delta_median': round(sub['delta'].median(), 3),
                'v2_wins': int((sub['delta'] > 0.01).sum()),
                'v3_wins': int((sub['delta'] < -0.01).sum()),
            })
        # accumulators
        acc = both[both['is_accumulator']]
        if len(acc) > 0:
            rows.append({
                'modality': 'accumulators', 'n': len(acc),
                'v2_median': round(acc['auc_v2'].median(), 3),
                'v3_median': round(acc['auc_v3'].median(), 3),
                'delta_median': round(acc['delta'].median(), 3),
                'v2_wins': int((acc['delta'] > 0.01).sum()),
                'v3_wins': int((acc['delta'] < -0.01).sum()),
            })

        sdf = pd.DataFrame(rows)
        print(sdf.to_string(index=False))
        save_result(sdf, 'M9_v2_vs_v3_summary', 'cell_m9')
        save_result(both.sort_values('delta', ascending=False),
                    'M9_v2_vs_v3_per_model', 'cell_m9')

        # bar chart
        mods_plot = [r for r in rows if r['modality'] != 'other']
        if mods_plot:
            labels = [r['modality'] for r in mods_plot]
            v2_vals = [r['v2_median'] for r in mods_plot]
            v3_vals = [r['v3_median'] for r in mods_plot]
            x = np.arange(len(labels))
            w = 0.35
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.bar(x - w/2, v2_vals, w, label='V2 buildings (zonal)', color='#1f77b4')
            ax.bar(x + w/2, v3_vals, w, label='V3 points (pixel)', color='#ff7f0e')
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
            ax.set_ylabel('Median AUC')
            ax.set_title('NB11b M9: V2 vs V3 Median AUC by Modality + Accumulators')
            ax.axhline(0.5, color='grey', ls='--', lw=0.8)
            ax.legend()
            for i_b in range(len(labels)):
                d = v2_vals[i_b] - v3_vals[i_b]
                top = max(v2_vals[i_b], v3_vals[i_b]) + 0.01
                ax.text(x[i_b], top, f'{d:+.2f}', ha='center', fontsize=8, fontweight='bold',
                        color='#1f77b4' if d > 0 else '#ff7f0e')
            plt.tight_layout()
            save_fig(fig, 'M9_v2_vs_v3_chart', 'cell_m9')

        # per-city: best model in each group, side by side
        print("\n  Per-city best model comparison:")
        for city in sorted(CITIES_TO_PROCESS):
            for grp_name, grp_oof in groups_loaded.items():
                ms = model_sanity_all[grp_name]
                headline = ms[(~ms['is_leaky']) & (~ms['is_degenerate'])]
                best_mid = headline.iloc[0]['model_id'] if len(headline) > 0 else None
                if best_mid is None:
                    continue
                cdf = grp_oof[(grp_oof['model_id'] == best_mid) & (grp_oof['city'] == city)]
                cm = cdf['cm_class'].value_counts()
                tp = int(cm.get('TP', 0)); fn = int(cm.get('FN', 0))
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                print(f"    {city:20s} [{grp_name:15s}] {best_mid:40s} recall={recall:.2f} "
                      f"TP={tp} FN={fn}")


# CELL M10: SUMMARY
All outputs, key findings, pointers to NB10.

In [ ]:
# @title CELL M10: SUMMARY
print("=" * 70)
print("NB11b v5 SUMMARY")
print("=" * 70)

for grp_name in groups_loaded:
    ms = model_sanity_all[grp_name]
    headline = ms[(~ms['is_leaky']) & (~ms['is_degenerate'])]
    acc = headline[headline['is_accumulator']]
    print(f"\n  === {grp_name} ===")
    print(f"  Models: {len(headline)} headline, {len(acc)} accumulators")
    if len(headline) > 0:
        best = headline.iloc[0]
        print(f"  Best overall: {best['model_id']}  AUC={best['auc']:.3f}  [{best['modality_family']}]")
    if len(acc) > 0:
        best_acc = acc.sort_values('auc', ascending=False).iloc[0]
        print(f"  Best accumulator: {best_acc['model_id']}  AUC={best_acc['auc']:.3f}")

    for fam in ['SAR', 'MS', 'multimodal']:
        sub = headline[headline['modality_family'] == fam]
        if len(sub) > 0:
            b = sub.sort_values('auc', ascending=False).iloc[0]
            print(f"  Best {fam}: {b['model_id']}  AUC={b['auc']:.3f}")

print(f"\n  Key findings for NB10:")
print(f"  1. SAR signal needs zonal stats (V2); MS works at pixel level (V3)")
print(f"  2. Accumulators are the strongest single-modality SAR features")
print(f"  3. Per-city error patterns vary by damage density, not model quality")
print(f"  4. No model achieves balanced TP+TN without per-city threshold tuning")
print(f"\n  Output: {OUT_DIR}")

summary = {
    'notebook': 'NB11b_v5',
    'groups': list(groups_loaded.keys()),
    'n_models': {g: len(model_sanity_all[g]) for g in groups_loaded},
}
save_result(summary, 'nb11b_summary', 'cell_m10', fmt='json')
print(f"\n  Done.")
